In [1]:
import sys
import os
sys.path.append(os.path.abspath('..')) # Даем доступ к папке src

In [2]:
# Импортируем НАШИ классы из папки src
from src.dgp import DataGenerator
from src.models import BaselineDiD, CausalImpactBSTS

In [3]:
TRUE_EFFECT = 15.0
T_PRE = 60

In [4]:
print("=== ТЕСТ H1: Нарушение трендов ===")
dgp_trends = DataGenerator(t_pre=T_PRE)
df_trends = dgp_trends.get_scenario_non_parallel_trends(effect_size=TRUE_EFFECT)

=== ТЕСТ H1: Нарушение трендов ===


In [5]:
did_model = BaselineDiD()
did_effect = did_model.fit_predict(df_trends)

In [6]:
import pandas.core.dtypes.common as c
print("patched:", hasattr(c, "is_datetime_or_timedelta_dtype"))

patched: True


In [7]:
bsts_model = CausalImpactBSTS(t_pre=T_PRE)
bsts_effect, _, _ = bsts_model.fit_predict(df_trends)

C:\Users\Gloccium\Documents\MastersProject\MastersDiplomaProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\representation.py:374: FutureWarning: Unknown keyword arguments: dict_keys(['alpha']).Passing unknown keyword arguments will raise a TypeError beginning in version 0.15.
  warnings.warn(msg, FutureWarning)


In [8]:
print(f"Истинный эффект: {TRUE_EFFECT}")
print(f"DiD оценка: {did_effect:.2f} | Смещение (Bias): {abs(did_effect - TRUE_EFFECT):.2f}")
print(f"BSTS оценка: {bsts_effect:.2f} | Смещение (Bias): {abs(bsts_effect - TRUE_EFFECT):.2f}")
print("-" * 30)

Истинный эффект: 15.0
DiD оценка: 46.93 | Смещение (Bias): 31.93
BSTS оценка: 26.78 | Смещение (Bias): 11.78
------------------------------


In [9]:
print("=== ТЕСТ H2: Сезонность и интервалы ===")
dgp_seas = DataGenerator(t_pre=T_PRE)
df_seas = dgp_seas.get_scenario_seasonality(effect_size=TRUE_EFFECT)

=== ТЕСТ H2: Сезонность и интервалы ===


In [10]:
bsts_model_s = CausalImpactBSTS(t_pre=T_PRE)
bsts_effect_s, ci_lower, ci_upper = bsts_model_s.fit_predict(df_seas)

C:\Users\Gloccium\Documents\MastersProject\MastersDiplomaProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\representation.py:374: FutureWarning: Unknown keyword arguments: dict_keys(['alpha']).Passing unknown keyword arguments will raise a TypeError beginning in version 0.15.
  warnings.warn(msg, FutureWarning)


In [11]:
coverage = "ДА" if ci_lower <= TRUE_EFFECT <= ci_upper else "НЕТ"
print(f"Истинный эффект: {TRUE_EFFECT}")
print(f"BSTS оценка: {bsts_effect_s:.2f}")
print(f"BSTS Доверительный интервал: [{ci_lower:.2f}, {ci_upper:.2f}]")
print(f"Истинный эффект попал в интервал (Coverage): {coverage}")
print("-" * 30)

Истинный эффект: 15.0
BSTS оценка: 12.75
BSTS Доверительный интервал: [23.57, 1.93]
Истинный эффект попал в интервал (Coverage): НЕТ
------------------------------


In [12]:
print("=== ТЕСТ H3: Скорость вычислений ===")
print(f"Время DiD (прокси для SDID): {did_model.time_taken:.4f} сек")
print(f"Время BSTS (MCMC): {bsts_model_s.time_taken:.4f} сек")
print(f"BSTS медленнее в {bsts_model_s.time_taken / max(did_model.time_taken, 0.0001):.0f} раз")

=== ТЕСТ H3: Скорость вычислений ===
Время DiD (прокси для SDID): 0.0079 сек
Время BSTS (MCMC): 0.0437 сек
BSTS медленнее в 6 раз
